# Data curation with CuratorKIT — `DataManager` (training & eval)

`DataManager` runs every dataset split through [CuratorKIT](https://github.com/Lexsi-Labs/CuratorKIT) before
handing it to a trainer or evaluator: schema validation, optional text cleaning, optional
deduplication, and canonical-field normalization.

This notebook covers:
1. Installing CuratorKIT
2. Baseline `DataManager` usage (defaults)
3. The `curator_*` options and what each one does
4. Curating an in-memory dataset with duplicate/malformed rows
5. Curating a real Hugging Face dataset
6. Using the same options in **eval** (`aligntune.eval.runner.EvalConfig`)


## 1. Installing CuratorKIT

CuratorKIT is already a core dependency in `pyproject.toml`:

```
"curatorkit[connectors] @ git+https://github.com/Lexsi-Labs/CuratorKIT.git",
```

So a normal `pip install -e .` of this repo installs it automatically. If you're running this
notebook standalone (e.g. Colab) without an editable install, install it directly:


In [ ]:
%pip install -q "curatorkit[connectors] @ git+https://github.com/Lexsi-Labs/CuratorKIT.git"
import curatorkit
print("curatorkit version:", getattr(curatorkit, "__version__", "unknown"))

## 2. Baseline `DataManager` usage

`DataManager` always schema-gates rows by default (`curator_schema_gate=True`) even with no other
options set — this is what every trainer already gets today.

In [ ]:
from datasets import Dataset
from aligntune.data.manager import DataManager

baseline_rows = [
    {"prompt": "What is 2+2?", "completion": "4"},
    {"prompt": "Name the largest planet.", "completion": "Jupiter"},
]
baseline_ds = Dataset.from_list(baseline_rows)

manager = DataManager(task_type="sft", val_split_ratio=None)
result = manager.load_dataset(baseline_ds)
print(result)
print(result["train"][0])

## 3. The `curator_*` options

| Option | Default | Effect |
|---|---|---|
| `curator_schema_gate` | `True` | Reject rows that don't match the task's canonical schema (always runs) |
| `curator_clean` | `False` | Run CuratorKIT's `TextCleaner` over accepted rows |
| `curator_dedup` | `"none"` | `"exact"` runs `ExactDeduplicator`; `"none"` disables it |
| `curator_use_tiktoken` | `False` | Use tiktoken for schema token-length counting instead of whitespace count |
| `curator_max_tokens` | `1_000_000` | Max token length enforced by the schema gate |

These fields live on every training `DatasetConfig` (SFT/RL/ES/distill) **and**, as of this
notebook, on `aligntune.eval.runner.EvalConfig` — same names, same defaults, same behavior.

## 4. Curating an in-memory dataset (duplicates + a malformed row)

A deterministic example: two exact-duplicate pairs and one empty row.

In [ ]:
dirty_rows = [
    {"prompt": "What is the capital of France?", "completion": "The capital of France is Paris."},
    {"prompt": "What is the capital of France?", "completion": "The capital of France is Paris."},  # exact dup
    {"prompt": "Explain photosynthesis in one sentence.", "completion": "Photosynthesis converts light energy into chemical energy in plants."},
    {"prompt": "Explain photosynthesis in one sentence.", "completion": "Photosynthesis converts light energy into chemical energy in plants."},  # exact dup
    {"prompt": "Write a haiku about autumn.", "completion": "Leaves drift to the ground\nCool wind hums through empty trees\nAutumn settles in"},
    {"prompt": "", "completion": ""},  # malformed / empty row
]
dirty_ds = Dataset.from_list(dirty_rows)
print(f"Input: {len(dirty_ds)} rows")

curated_manager = DataManager(
    task_type="sft",
    val_split_ratio=None,
    curator_schema_gate=True,
    curator_clean=True,
    curator_dedup="exact",
)
curated = curated_manager.load_dataset(dirty_ds)["train"]

print(f"Output after curation: {len(curated)} rows")
for row in curated:
    print(" -", row["prompt"][:60])

assert len(curated) == 3, "expected the empty row and both duplicates to be removed" 

## 5. Curating a real Hugging Face dataset

Same options, this time on a small real slice of `tatsu-lab/alpaca`.

In [ ]:
from datasets import load_dataset

raw_slice = load_dataset("tatsu-lab/alpaca", split="train")
max_samples=50,
print(f"Raw slice: {len(raw_slice)} rows")

alpaca_manager = DataManager(
    task_type="sft",
    val_split_ratio=None,
    curator_schema_gate=True,
    curator_clean=True,
    curator_dedup="exact",
)
alpaca_curated = alpaca_manager.load_dataset(raw_slice)["train"]

print(f"Curated: {len(alpaca_curated)} rows")
print("Columns:", alpaca_curated.column_names)
print("Sample:", {k: str(v)[:60] for k, v in alpaca_curated[0].items() if k in ("prompt", "completion", "id")})

## 6. Using the same options in eval

`aligntune.eval.runner.EvalConfig` now carries the identical five `curator_*` fields, and
`run_eval(...)` forwards them into the `DataManager` it builds internally
(`aligntune/eval/runner.py`) — so an eval run curates its dataset exactly the way a training run
does.

Below mirrors what `run_eval` does internally, using the same dirty dataset from section 4 so the
result is easy to check.

In [ ]:
from aligntune.eval.runner import EvalConfig

eval_cfg = EvalConfig(
    model_path="unused-for-this-demo",
    output_dir="./eval_results",
    curator_schema_gate=True,
    curator_clean=True,
    curator_dedup="exact",
)

# Same construction `run_eval()` performs internally at the "Load Dataset" step.
eval_data_manager = DataManager(
    task_type="sft",
    val_split_ratio=None,
    curator_schema_gate=eval_cfg.curator_schema_gate,
    curator_clean=eval_cfg.curator_clean,
    curator_dedup=eval_cfg.curator_dedup,
    curator_use_tiktoken=eval_cfg.curator_use_tiktoken,
    curator_max_tokens=eval_cfg.curator_max_tokens,
)

eval_curated = eval_data_manager.load_dataset(dirty_ds)["train"]
print(f"Eval-path curated output: {len(eval_curated)} rows (matches section 4's result: {len(curated)})")
assert len(eval_curated) == len(curated)

## Summary

- `curator_schema_gate` / `curator_clean` / `curator_dedup` / `curator_use_tiktoken` /
  `curator_max_tokens` are the config surface — same names on every training `DatasetConfig` and on
  `eval.runner.EvalConfig`.
- CuratorKIT must be installed (`pyproject.toml` main dependency, see section 1).
- **Known caveat**: `DataManager._run_curator` (`aligntune/data/manager.py`) currently resolves
  `dedup` with `"exact" if self.curator_dedup else "none"`. Since `curator_dedup` is now a
  *string* (`"none"` or `"exact"`) rather than a bool, any non-empty string — including `"none"`
  — is truthy, so dedup currently always runs as `"exact"` once CuratorKIT is exercised. This is a
  pre-existing issue in the shared `DataManager` code, not specific to eval.